# Chapter 11 — Context Management in Production Systems · Practice

**Goal:** build every context-management technique from the chapter as code you can inspect and measure: a case-facts block, tool-result trimming, position-aware synthesis input, scratchpads, subagent hand-offs, crash-safe state, session decisions, token budgets, server-side compaction, context editing, the memory tool and cache-friendly layouts. The mini-project combines them into a long-running support conversation manager.

- **Theory:** [README.md](README.md) (read the matching section before each part)
- **Estimated time:** 3–4 hours
- **Estimated API cost:** probably under $0.15 with the default model (`claude-sonnet-5-5`). This is an estimate from about 15 small live calls at list prices, not a measured figure; check `usage` after your first live cell. Token counting is free. Everything else runs offline.
- **Prerequisites:** [00 — Prerequisites](../../00-prerequisites/README.md) (venv + `.env`), [Chapter 1](../01-claude-api-fundamentals/README.md) (statelessness, `usage`), [Chapter 2](../02-tool-use/README.md) (the tool loop), [Chapter 3](../03-agent-sdk/README.md) (hooks, subagents, sessions)

| Marker | Meaning |
|---|---|
| **(offline)** | Runs without a key. Many offline cells use `ScriptedClaude`: a **real** `anthropic.Anthropic` client whose HTTP layer replays canned responses, so the SDK's own request building, parsing and types run for real. |
| **(live)** | Calls the API. Skip these if you have no credit; no offline cell depends on them. |

This notebook does not run the Claude Agent SDK or the `claude` CLI. Where the README shows Agent SDK code (hooks, subagents, sessions), the notebook tests the part you write yourself (the hook callback, the subagent prompt, the decision logic) offline and shows the SDK wiring as a snippet.

In [ ]:
import asyncio
import copy
import json
import math
import os
import re
import tempfile
import urllib.parse
from concurrent.futures import ThreadPoolExecutor
from itertools import count
from pathlib import Path
from typing import Any, Callable

import anthropic
import httpx2
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, Field, ValidationError

load_dotenv(find_dotenv())

MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment; never hardcode it

WORKDIR = Path(tempfile.mkdtemp(prefix="ch11-"))  # every file this notebook writes goes here
print("Model:", MODEL, "| scratch dir:", WORKDIR)

In [ ]:
# Live cells need a key. Offline cells work without one, so skip this cell if you have no credit.
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see 00-prerequisites)"

### Offline toolkit (run this once)

- `ScriptedClaude(*replies)`: `.client` is a real SDK client. Each call to `/v1/messages` gets the next scripted reply; calls to `count_tokens` get an estimate. Every request (path, `anthropic-beta` header, JSON body) is recorded in `.requests`.
- `reply(...)`, `text(...)`, `tool_use(...)`: build response bodies in the Messages API wire format.
- `est_tokens(x)`: a **rough** offline estimate (about 4 characters per token). It is *not* Claude's tokenizer; real numbers come from `count_tokens` and `usage`.
- `run_async(coro)`: runs an async function (such as an Agent SDK hook callback) from a normal cell.

In [ ]:
_ids = count(1)


def est_tokens(obj: Any) -> int:
    """Rough offline estimate: ~4 characters per token. Not Claude's tokenizer (use count_tokens for real numbers)."""
    s = obj if isinstance(obj, str) else json.dumps(obj, ensure_ascii=False, default=str)
    return math.ceil(len(s) / 4)


def text(t: str) -> dict:
    return {"type": "text", "text": t}


def tool_use(name: str, tool_input: dict, id: str | None = None) -> dict:
    return {"type": "tool_use", "id": id or f"toolu_{next(_ids):04d}", "name": name, "input": tool_input}


def reply(*blocks: dict, stop_reason: str | None = None, usage: dict | None = None, **extra) -> dict:
    """A Messages API response body. stop_reason defaults to tool_use / end_turn based on the blocks."""
    if stop_reason is None:
        stop_reason = "tool_use" if any(b["type"] == "tool_use" for b in blocks) else "end_turn"
    return {"id": f"msg_{next(_ids):04d}", "type": "message", "role": "assistant", "model": "scripted",
            "content": list(blocks), "stop_reason": stop_reason, "stop_sequence": None,
            "usage": usage or {"input_tokens": 0, "output_tokens": 0}, **extra}


class ScriptedClaude:
    """Offline stand-in for the Messages API: replays scripted replies, records every request."""

    def __init__(self, *replies: dict):
        self.replies = list(replies)
        self.requests: list[dict] = []
        transport = httpx2.MockTransport(self._handle)
        self.client = anthropic.Anthropic(api_key="offline", max_retries=0,
                                          http_client=anthropic.DefaultHttpxClient(transport=transport))

    def _handle(self, request: httpx2.Request) -> httpx2.Response:
        body = json.loads(request.content)
        self.requests.append({"path": request.url.path, "beta": request.headers.get("anthropic-beta"), "body": body})
        if request.url.path.endswith("/count_tokens"):
            n = est_tokens([body.get("system"), body.get("tools"), body["messages"]])
            out = {"input_tokens": n}
            if "context_management" in body:  # the beta endpoint also reports the pre-edit size
                out = {"input_tokens": n // 2, "context_management": {"original_input_tokens": n}}
            return httpx2.Response(200, json=out)
        if not self.replies:
            return httpx2.Response(500, json={"type": "error", "error": {"type": "api_error", "message": "script exhausted"}})
        body = dict(self.replies.pop(0))
        return httpx2.Response(body.pop("_status", 200), json=body)


def run_async(coro):
    """Run a coroutine from a notebook cell (Jupyter already has a running event loop)."""
    with ThreadPoolExecutor(1) as pool:
        return pool.submit(asyncio.run, coro).result()


def text_of(response) -> str:
    return "".join(b.text for b in response.content if b.type == "text")


print("offline toolkit ready")

### The running example

One support case runs through most sections: customer `CUST-58214` has **two issues**, a damaged blender on order `ORD-30417` ($74.95, ordered 2025-01-15, the customer wants a full refund and not store credit, and says an agent promised 25% off the next order) and a parcel for order `ORD-67911` sent to an old address. `lookup_order` returns **43 fields** per call, like many real order systems.

In [ ]:
ORDER_FIELDS = [
    "order_id", "customer_id", "status", "total", "currency", "order_date", "items", "return_eligible",
    "return_window_ends", "subtotal", "tax", "shipping_cost", "discount_code", "discount_amount", "payment_method",
    "payment_last4", "payment_auth_id", "billing_address", "shipping_address", "shipping_method", "carrier",
    "tracking_number", "tracking_events", "warehouse_id", "picker_id", "packed_at", "shipped_at", "delivered_at",
    "gift_wrap", "gift_message", "loyalty_points_earned", "loyalty_tier", "marketing_opt_in", "utm_source",
    "utm_campaign", "device", "ip_country", "fraud_score", "risk_flags", "invoice_url", "notes_internal",
    "created_at", "updated_at",
]
assert len(ORDER_FIELDS) == 43


def make_order(order_id: str, total: float, order_date: str, item: str, status: str, address: str) -> dict:
    """A realistic, verbose order record: 43 fields, most of them irrelevant to a return decision."""
    events = [{"ts": f"{order_date}T{8 + i:02d}:00:00Z", "location": f"Hub {i}", "code": f"EV{i:03d}",
               "description": "Package processed at sorting facility"} for i in range(8)]
    return {
        "order_id": order_id, "customer_id": "CUST-58214", "status": status, "total": total, "currency": "USD",
        "order_date": order_date, "items": [{"sku": "SKU-" + item[:3].upper(), "name": item, "qty": 1, "price": total}],
        "return_eligible": True, "return_window_ends": "2025-02-14", "subtotal": round(total / 1.08, 2),
        "tax": round(total - total / 1.08, 2), "shipping_cost": 0.0, "discount_code": None, "discount_amount": 0.0,
        "payment_method": "card", "payment_last4": "4242", "payment_auth_id": "auth_9f8e7d6c5b4a",
        "billing_address": {"line1": "7 Oak Avenue", "city": "Springfield", "zip": "11111", "country": "US"},
        "shipping_address": {"line1": address, "city": "Springfield", "zip": "11111", "country": "US"},
        "shipping_method": "standard", "carrier": "ParcelCo", "tracking_number": "PC" + order_id[-5:] + "XX99",
        "tracking_events": events, "warehouse_id": "WH-EAST-3", "picker_id": "EMP-5531",
        "packed_at": f"{order_date}T06:12:00Z", "shipped_at": f"{order_date}T07:40:00Z", "delivered_at": None,
        "gift_wrap": False, "gift_message": None, "loyalty_points_earned": int(total), "loyalty_tier": "silver",
        "marketing_opt_in": True, "utm_source": "newsletter", "utm_campaign": "winter-sale", "device": "ios-app",
        "ip_country": "US", "fraud_score": 0.02, "risk_flags": [], "invoice_url": f"https://example.com/inv/{order_id}",
        "notes_internal": "Customer contacted support previously about a different order. No action needed.",
        "created_at": f"{order_date}T05:59:00Z", "updated_at": f"{order_date}T07:40:00Z",
    }


ORDERS = {
    "ORD-30417": make_order("ORD-30417", 74.95, "2025-01-15", "Blender", "delivered", "7 Oak Avenue"),
    "ORD-67911": make_order("ORD-67911", 34.50, "2025-01-20", "Kettle", "in_transit", "3 Old Mill Road"),
}
print(len(ORDERS["ORD-30417"]), "fields, about", est_tokens(ORDERS["ORD-30417"]), "tokens per lookup_order result (estimate)")

## 11.1 Extract facts into a persistent case-facts block

The API is stateless, so every request must carry the history you want Claude to know. When the history grows you summarize it, and prose summaries keep the *topic* while losing the *value*: "25% off" becomes "a discount came up". The fix is a separate, structured **case-facts** layer, one entry per issue, that is merged from tool results and extracted claims, never summarized, and sent with every request.

[Theory](README.md#111-extract-facts-into-a-persistent-case-facts-block)

### What a summary keeps (offline)

In [ ]:
EARLY_TURNS = [  # what the customer said in turns 3-9
    "The blender from order ORD-30417 arrived damaged. I ordered it on 2025-01-15.",
    "I want a full refund of $74.95, not store credit.",
    "Your colleague promised me a 25% discount on my next order.",
]
EXACT_VALUES = ["ORD-30417", "2025-01-15", "$74.95", "not store credit", "25%"]

# The kind of summary README 11.1 describes ("A discount came up"), written many turns later:
PROSE_SUMMARY = ("The customer reported a damaged kitchen appliance and wants compensation. "
                 "A discount came up, and there was a delivery delay.")


def surviving(values: list[str], text_: str) -> dict[str, bool]:
    return {v: v in text_ for v in values}


for v, ok in surviving(EXACT_VALUES, PROSE_SUMMARY).items():
    print(f"{'✓' if ok else '✗'} {v}")

> **Observe:** none of the values the next turn depends on survive. "Wants compensation" is not "full refund, not store credit", and the agent will now offer store credit with confidence.

**(live)** Ask Claude for a short summary and test which values survive. Then rerun with a "preserve every number" instruction. Even when the second summary keeps them, you are still trusting a lossy step with the facts a refund depends on.

In [ ]:
transcript = "\n".join(f"Customer: {t}\nAgent: Thanks, I've noted that." for t in EARLY_TURNS) + \
    "\nCustomer: Also, the box was soaked and the manual is missing.\nAgent: Sorry about that, I'll include it in the case."

for label, instruction in [("plain", "Summarize this support conversation in two sentences."),
                           ("preserve", "Summarize this support conversation in two sentences. "
                                        "Preserve every order ID, amount, percentage, date and customer request verbatim.")]:
    r = client.messages.create(model=MODEL, max_tokens=400,
                               messages=[{"role": "user", "content": f"<transcript>\n{transcript}\n</transcript>\n\n{instruction}"}])
    summary = text_of(r)
    kept = surviving(EXACT_VALUES, summary)
    print(f"[{label}] kept {sum(kept.values())}/{len(kept)}: {summary.strip()[:300]}\n")

### A typed case-facts model (offline)

A typed version of the README's case-facts block, with two helpers: `issue(n)` creates or returns an issue entry, and `bump()` increments the version each time the block changes. Each fact records **where it came from** and **when**, and a customer claim is marked `verified=False` until a system confirms it.

In [ ]:
class Fact(BaseModel):
    value: str
    source: str                 # "lookup_order", "get_customer", "customer"
    turn: int
    verified: bool = True       # False for customer claims not yet checked against a system


class Issue(BaseModel):
    issue_id: int
    facts: dict[str, Fact] = Field(default_factory=dict)
    conflicts: list[str] = Field(default_factory=list)

    def upsert(self, key: str, new: Fact) -> None:
        old = self.facts.get(key)
        if old and old.value != new.value:
            # Never overwrite silently: record the disagreement and keep the verified value
            self.conflicts.append(f"{key}: {old.value} ({old.source}, t{old.turn}) vs {new.value} ({new.source}, t{new.turn})")
            if old.verified and not new.verified:
                return
        self.facts[key] = new


class CaseFacts(BaseModel):
    customer_id: Fact | None = None
    issues: list[Issue] = Field(default_factory=list)
    version: int = 0

    def issue(self, issue_id: int) -> Issue:
        for i in self.issues:
            if i.issue_id == issue_id:
                return i
        self.issues.append(Issue(issue_id=issue_id))
        return self.issues[-1]

    def bump(self) -> None:
        self.version += 1

    def render(self) -> str:
        lines = [f'<case_facts version="{self.version}">']
        if self.customer_id:
            lines.append(f"customer_id: {self.customer_id.value} (source: {self.customer_id.source})")
        for issue in self.issues:
            lines.append(f"- issue {issue.issue_id}:")
            for key, f in issue.facts.items():
                tag = "" if f.verified else ", UNVERIFIED claim"
                lines.append(f"    {key}: {f.value} ({f.source}, turn {f.turn}{tag})")
            for c in issue.conflicts:
                lines.append(f"    CONFLICT: {c}")
        lines.append("</case_facts>")
        return "\n".join(lines)

**Rule 1: copy values from tool results in code.** The order total is data the tool returned, so code copies it. Only things that exist solely in the customer's words go through an extraction step (in production, a structured-output call; here the extracted claims are scripted), and they are stored as unverified claims.

In [ ]:
def ingest_order(facts: CaseFacts, issue_id: int, order: dict, turn: int) -> None:
    """Deterministic: copy exact values from the tool result. No model involved."""
    issue = facts.issue(issue_id)
    for key, value in [("order_id", order["order_id"]), ("amount", f"{order['total']:.2f} {order['currency']}"),
                       ("order_date", order["order_date"]), ("order_status", order["status"])]:
        issue.upsert(key, Fact(value=value, source="lookup_order", turn=turn))
    facts.bump()


def ingest_claim(facts: CaseFacts, issue_id: int, key: str, value: str, turn: int) -> None:
    """Something only the customer said: store it, labeled as an unverified claim."""
    facts.issue(issue_id).upsert(key, Fact(value=value, source="customer", turn=turn, verified=False))
    facts.bump()


facts = CaseFacts(customer_id=Fact(value="CUST-58214", source="get_customer", turn=2))
ingest_order(facts, 1, ORDERS["ORD-30417"], turn=3)
ingest_claim(facts, 1, "customer_request", "full refund, NOT store credit", turn=4)
ingest_claim(facts, 1, "promised", "25% discount on next order", turn=9)
ingest_order(facts, 2, ORDERS["ORD-67911"], turn=10)
ingest_claim(facts, 2, "new_address", "42 Elm Street, Springfield", turn=11)
ingest_claim(facts, 1, "amount", "99.99 USD", turn=22)   # the customer misremembers the amount later on

print(facts.render())

> **Observe:** the turn-22 claim of $99.99 did **not** replace the verified $74.95 from `lookup_order`; it became a `CONFLICT` line the agent can raise with the customer. Issue 1 and issue 2 keep separate `order_id` / `amount` / `order_status`, so nothing leaks between them (S2).

> **Try it:** ingest a *verified* correction for issue 2 (a second `lookup_order` result with `status="delivered"`). What does `upsert` do when the new value is verified?

### Where to put the block: three placements (offline)

All three options send the facts on every request. They differ in what they do to the **cached prefix**. The helper below serializes a request in cache order (tools → system → messages) and measures how much of the previous request's prefix the next request leaves unchanged. Unchanged prefix is what prompt caching can reuse; it is also what keeps earlier thinking blocks valid on current models.

In [ ]:
POLICY = "You are a support agent for an online store. Refund policy: damaged items qualify for a refund within 30 days. " * 20
HISTORY = [  # a few earlier turns, exactly as they were sent
    {"role": "user", "content": "Hi, my blender arrived damaged."},
    {"role": "assistant", "content": "Sorry to hear that. Which order is it?"},
]


def cache_units(req: dict) -> list[str]:
    """Serialize a request in prompt-cache order: each tool, then the system prompt, then each message."""
    units = [json.dumps(t, sort_keys=True) for t in req.get("tools", [])]
    units.append("SYSTEM " + json.dumps(req.get("system", ""), sort_keys=True))
    return units + [json.dumps(m, sort_keys=True) for m in req["messages"]]


def unchanged_prefix(prev: dict, cur: dict) -> int:
    """Estimated tokens at the start of `cur` that are byte-identical to `prev`."""
    n = 0
    for a, b in zip(cache_units(prev), cache_units(cur)):
        if a != b:
            break
        n += est_tokens(a)
    return n


def in_system(history, facts_block, msg):            # option 1: re-render system each turn
    return {"system": POLICY + "\n" + facts_block, "messages": history + [{"role": "user", "content": msg}]}


def in_newest_turn(history, facts_block, msg):       # option 2: portable, append-only
    return {"system": POLICY, "messages": history + [{"role": "user", "content": [text(facts_block), text(msg)]}]}


def turn_scoped_system(history, facts_block, msg):   # option 3: beta clear_at on supported models
    return {"system": POLICY, "messages": history + [{"role": "user", "content": msg},
                                                     {"role": "system", "clear_at": "next_user_message", "content": facts_block}]}


for name, build in [("in system", in_system), ("newest user turn", in_newest_turn), ("turn-scoped system msg", turn_scoped_system)]:
    turn1 = build(HISTORY, facts.render(), "It's order ORD-30417.")
    hist2 = turn1["messages"] + [{"role": "assistant", "content": "Thanks, checking now."}]
    facts.bump()  # facts change between turns
    turn2 = build(hist2, facts.render(), "Any update?")
    total = est_tokens("".join(cache_units(turn2)))
    print(f"{name:<24} turn 2 reuses ~{unchanged_prefix(turn1, turn2):>4} of ~{total} tokens")

> **Observe:** re-rendering `system` changes the prefix at the very start, so turn 2 reuses almost nothing. The newest-turn option leaves the whole earlier request unchanged (the old facts copy stays in history, so tell the model "the latest `case_facts` block supersedes earlier ones"). The turn-scoped system message also leaves history unchanged, and once a newer user message exists the older copy is cleared: it renders nothing and costs no input tokens.

Turn-scoped messages need the `mid-conversation-system-clear-at-2026-08-21` beta header. **Authority:** Claude treats system text as operator instructions, so a system message suits only verified, system-sourced facts. The block above also carries customer claims (marked UNVERIFIED); in production, split it: verified facts in the system message, customer claims in the newest user turn.

**(live)** Send the turn-scoped option for real. The next turn will not pay for this block again. (This demo block also carries customer claims; in production those stay in the user turn.)

In [ ]:
req = turn_scoped_system(HISTORY, facts.render(), "What refund amount and type did I ask for? One sentence.")
r = client.beta.messages.create(model=MODEL, max_tokens=300, betas=["mid-conversation-system-clear-at-2026-08-21"], **req)
print(r.stop_reason, "|", text_of(r).strip())
print("usage:", r.usage.input_tokens, "in /", r.usage.output_tokens, "out")

## 11.2 Trim verbose tool results

Every `tool_result` stays in history and is re-sent, and re-billed, on every later turn. Trim **once, at ingest**: in the tool itself, in your loop, or in a `PostToolUse` hook for tools you do not own. Keep the raw record outside the window with a reference to it, so nothing is lost. Between agents, the same idea becomes an **output contract**: upstream agents return structured facts, not raw pages.

[Theory](README.md#112-trim-verbose-tool-results)

In [ ]:
RETURN_FIELDS = ("order_id", "status", "total", "currency", "order_date",
                 "items", "return_eligible", "return_window_ends")
RAW_STORE: dict[str, dict] = {}            # full payloads live outside the context window


def trim_order(raw: dict) -> dict:
    """Keep only what a return/refund decision needs, plus a pointer to the full record."""
    slim = {k: raw[k] for k in RETURN_FIELDS if k in raw}
    RAW_STORE[raw["order_id"]] = raw       # nothing is lost: code (or a get_order_detail tool) can still read it
    slim["full_record_ref"] = f"order_raw/{raw['order_id']}"
    return slim


def tool_result_block(tool_use_id: str, raw: dict) -> dict:
    return {"type": "tool_result", "tool_use_id": tool_use_id, "content": json.dumps(trim_order(raw))}


raw = ORDERS["ORD-30417"]
slim = trim_order(raw)
print(f"raw: {len(raw)} fields ~{est_tokens(raw)} tokens  ->  trimmed: {len(slim)} fields ~{est_tokens(slim)} tokens")
print(json.dumps(slim, indent=1)[:400])

### Why it matters: the result is re-sent on every later turn (offline)

A 30-turn case that calls `lookup_order` 6 times. Each result enters history at its turn and is billed again on every turn after it.

In [ ]:
CALL_TURNS = [3, 6, 10, 14, 18, 22]
TURNS = 30


def cumulative_tool_tokens(tokens_per_result: int) -> int:
    """Total input tokens spent on tool results over the whole conversation."""
    return sum(tokens_per_result * sum(1 for t in CALL_TURNS if t <= turn) for turn in range(1, TURNS + 1))


raw_total, slim_total = cumulative_tool_tokens(est_tokens(raw)), cumulative_tool_tokens(est_tokens(slim))
print(f"tool-result tokens billed over {TURNS} turns: raw ~{raw_total:,}  trimmed ~{slim_total:,}  "
      f"({1 - slim_total / raw_total:.0%} less)")

> **Observe:** the saving compounds. A field you keep is paid for once per remaining turn, which is why "the model might need it later" is the wrong default: store it outside the window and keep a reference.

### The same trimming as a `PostToolUse` hook (offline harness)

For an MCP tool you cannot change, the Agent SDK lets a `PostToolUse` hook replace what Claude sees via `hookSpecificOutput.updatedToolOutput`. The callback is plain async Python, so you can test it without running an agent. `input_data` below has the shape of the SDK's `PostToolUseHookInput` (`tool_name`, `tool_input`, `tool_response`, `tool_use_id`).

The SDK types `tool_response` as `Any`, and its shape depends on the tool. The harness below uses an MCP-style result with `content` blocks. Print `tool_response` once in a debug hook to confirm the shape for your tool, and return the replacement in the same shape: an MCP replacement is not schema-validated, so a wrong shape fails silently.

In [ ]:
def _payload(tool_response) -> dict:
    """MCP-style result: {"content": [{"type": "text", "text": "<json>"}], ...}. Normalize to a dict."""
    return json.loads("".join(b.get("text", "") for b in tool_response["content"]))


async def trim_lookup_order(input_data, tool_use_id, context):
    slim = trim_order(_payload(input_data["tool_response"]))
    return {"hookSpecificOutput": {
        "hookEventName": "PostToolUse",
        # Same shape as received. structuredContent is dropped on purpose: it would carry the full payload.
        "updatedToolOutput": {"content": [{"type": "text", "text": json.dumps(slim)}]},
    }}


tool_response = {"content": [{"type": "text", "text": json.dumps(ORDERS["ORD-30417"])}],
                 "structuredContent": ORDERS["ORD-30417"]}
fake_input = {"hook_event_name": "PostToolUse", "tool_name": "mcp__support__lookup_order", "tool_use_id": "toolu_42",
              "tool_input": {"order_id": "ORD-30417"}, "tool_response": tool_response}
out = run_async(trim_lookup_order(fake_input, "toolu_42", None))["hookSpecificOutput"]["updatedToolOutput"]
assert json.loads(out["content"][0]["text"])["order_id"] == "ORD-30417"
print(f"tool_response ~{est_tokens(tool_response)} tokens -> updatedToolOutput ~{est_tokens(out)} tokens")

Wiring it into an agent (not run here) looks like this:

```python
from claude_agent_sdk import ClaudeAgentOptions, HookMatcher

options = ClaudeAgentOptions(
    hooks={"PostToolUse": [HookMatcher(matcher="mcp__support__lookup_order", hooks=[trim_lookup_order])]},
)
```

### Trim before the first send, never afterwards (offline)

On Fable 5.1, Opus 5.5 and Sonnet 5.5, earlier thinking blocks are tied to the exact conversation prefix they were produced with. Editing an old `tool_result` in place changes that prefix for **every later turn**. The helper finds the first message that differs between what you sent and what you would send after an in-place edit.

In [ ]:
def first_changed_message(before: list[dict], after: list[dict]) -> int | None:
    for i, (a, b) in enumerate(zip(before, after)):
        if json.dumps(a, sort_keys=True) != json.dumps(b, sort_keys=True):
            return i
    return None if len(before) == len(after) else min(len(before), len(after))


sent = [
    {"role": "user", "content": "Refund status for ORD-30417?"},
    {"role": "assistant", "content": [{"type": "thinking", "thinking": "...", "signature": "sig-1"},
                                      tool_use("lookup_order", {"order_id": "ORD-30417"}, id="toolu_a")]},
    {"role": "user", "content": [{"type": "tool_result", "tool_use_id": "toolu_a", "content": json.dumps(raw)}]},
    {"role": "assistant", "content": [{"type": "thinking", "thinking": "...", "signature": "sig-2"}, text("It was delivered.")]},
    {"role": "user", "content": "OK, and the refund?"},
    {"role": "assistant", "content": [{"type": "thinking", "thinking": "...", "signature": "sig-3"}, text("Pending approval.")]},
]
edited = copy.deepcopy(sent)
edited[2]["content"][0]["content"] = json.dumps(trim_order(raw))  # "clean up" the old result later

i = first_changed_message(sent, edited)
later_thinking = [m for m in sent[i:] if m["role"] == "assistant"]
print(f"first changed message: {i}; assistant turns after it whose thinking is now invalid: {len(later_thinking)}")

> **Observe:** one late edit invalidates the thinking in every later assistant turn (and on accounts created on or after 2026-08-31 the API rejects that replay by default). Trim at ingest; to drop old results later, use server-side context editing (11.10).

### Trimming at agent boundaries: an output contract (offline)

A web-search agent that forwards raw pages, and a document agent that forwards its reasoning, can hand a synthesis agent well over 100K tokens. Change the **upstream contract** instead of adding a summarization hop: key facts, a short supporting quote, source metadata and a relevance score.

In [ ]:
class ResearchFinding(BaseModel):
    claim: str
    quote: str = Field(max_length=300)     # a short supporting quote, not the page
    source: str
    date: str
    relevance: float = Field(ge=0, le=1)


class AgentReport(BaseModel):
    agent: str
    findings: list[ResearchFinding]
    coverage: list[str]
    gaps: list[str]


raw_pages = "Lorem ipsum industry report text with charts, navigation menus and footers. " * 3200  # ~61K tokens
report = AgentReport(agent="web-search", coverage=["studio adoption"], gaps=["licensing revenue"], findings=[
    ResearchFinding(claim=f"Finding {i}: AI tools are used in {10 + i}% of surveyed studios",
                    quote="...the survey found adoption rising year over year...", source=f"https://example.org/r{i}",
                    date="2024-11-02", relevance=0.9 - i * 0.05) for i in range(12)])
print(f"raw forwarding ~{est_tokens(raw_pages):,} tokens  vs  structured report ~{est_tokens(report.model_dump()):,} tokens")
try:
    ResearchFinding(claim="x", quote="y" * 5000, source="s", date="d", relevance=0.5)
except ValidationError as e:
    print("contract rejects page dumps:", e.errors()[0]["msg"])

## 11.3 Position-aware input: lost in the middle

Models use the beginning and end of a long input most reliably and are more likely to miss findings in the middle. Fix the layout, not the content: a **key-findings summary first**, **explicit section headers** (XML `<document>` blocks) for the details, and the **task last**. Require subagents to send the metadata (source, date, method) that synthesis needs to attribute and weigh each finding.

[Theory](README.md#113-position-aware-input-lost-in-the-middle)

In [ ]:
FILLER = "Background discussion, methodology notes and tables that support the finding. " * 25
FINDINGS = [
    {"source": f"subagent-{i}", "date": f"2024-0{i + 1}-15", "method": m, "key_points": [kp], "detail": FILLER + kp + " " + FILLER}
    for i, (m, kp) in enumerate([
        ("survey", "Studios using AI composition tools grew from 12% to 31%."),
        ("interviews", "Session musicians report 20% fewer bookings for jingles."),
        ("market data", "Licensing revenue for stock music fell 8% year over year."),
        ("case study", "One label cut demo production time from 3 weeks to 4 days."),
        ("legal review", "Three pending lawsuits concern training data for music models."),
        ("survey", "62% of listeners could not tell AI-assisted tracks apart."),
    ])
]
QUESTION = "How is AI changing the music industry's labor market and revenue?"


def build_synthesis_input(findings: list[dict], question: str) -> str:
    """findings: [{"source", "date", "method", "key_points": [...], "detail": "..."}] from subagents."""
    summary = "\n".join(
        f"- [{i}] {pt} ({f['source']}, {f['date']})"
        for i, f in enumerate(findings, 1) for pt in f["key_points"]
    )
    sections = "\n".join(
        f'<document index="{i}">\n<source>{f["source"]}</source>\n<date>{f["date"]}</date>\n'
        f'<method>{f["method"]}</method>\n<document_content>\n{f["detail"]}\n</document_content>\n</document>'
        for i, f in enumerate(findings, 1)
    )
    return (
        f"<key_findings>\n{summary}\n</key_findings>\n\n"          # top: everything important, briefly
        f"<detailed_results>\n{sections}\n</detailed_results>\n\n"  # middle: navigable by headers
        f"<task>\n{question}\nCite findings by [index]. Cover every key finding listed above.\n</task>"  # end
    )


def build_naive_input(findings: list[dict], question: str) -> str:
    return question + "\n\n" + "\n\n".join(f["detail"] for f in findings)   # question first, everything else a wall


def first_mention_positions(prompt: str, findings: list[dict]) -> list[float]:
    """Relative position (0 = start, 1 = end) where each key point first appears."""
    return [prompt.find(f["key_points"][0]) / len(prompt) for f in findings]


for name, prompt in [("naive", build_naive_input(FINDINGS, QUESTION)), ("structured", build_synthesis_input(FINDINGS, QUESTION))]:
    pos = first_mention_positions(prompt, FINDINGS)
    middle = sum(0.25 < p < 0.75 for p in pos)
    print(f"{name:<11} first mentions at {[f'{p:.0%}' for p in pos]}  -> {middle} of {len(pos)} only in the middle; "
          f"question at the end: {prompt.rstrip().endswith('</task>')}")

> **Observe:** in the naive layout, the findings in the middle documents are first mentioned in the middle of the input, which is the position the model is most likely to skip. In the structured layout every finding appears in the first few percent (the summary), the details are still there under headers, and the task sits at the end.

### Required metadata (offline)

Make `source`, `date` and `method` required fields of each subagent's structured output. A finding without them cannot be dated or attributed at synthesis time, so reject it at the boundary.

In [ ]:
class SubagentFinding(BaseModel):
    source: str
    date: str
    method: str
    key_points: list[str] = Field(min_length=1)
    detail: str


try:
    SubagentFinding(source="subagent-x", method="survey", key_points=["AI adoption is 40%"], detail="...")
except ValidationError as e:
    print("rejected:", [(err["loc"][0], err["msg"]) for err in e.errors()])
validated = [SubagentFinding(**f) for f in FINDINGS]
print("accepted", len(validated), "findings with full metadata")

**(live)** Ask Claude to answer from the structured input and check which findings it cites.

In [ ]:
r = client.messages.create(model=MODEL, max_tokens=800,
                           messages=[{"role": "user", "content": build_synthesis_input(FINDINGS, QUESTION)}])
answer = text_of(r)
cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer)})
print("cited findings:", cited, "of", list(range(1, len(FINDINGS) + 1)))
print(answer[:600])

> **Try it:** send `build_naive_input` instead and compare the citations. With only ~6K tokens the effect is small; it grows with input length, which is the point of the layout rules.

## 11.4 Scratchpad files

In a long investigation, early specifics get buried under later exploration and answers drift to "typical patterns". A **scratchpad file** moves confirmed findings out of the conversation onto disk, where they survive `/compact`, `/clear` and new sessions. It works only if every finding carries **evidence** (a file and line, or a tool-result reference), the agent **reads it before answering**, and it stays a small, curated index.

[Theory](README.md#114-scratchpad-files)

In [ ]:
EVIDENCE = re.compile(r"[\w./-]+\.\w+:\d+|tool_result:\S+")


class Scratchpad:
    """An investigation scratchpad: confirmed findings (with evidence), open questions, decisions."""

    def __init__(self, path: Path):
        self.path = path
        self.findings: dict[str, str] = {}
        self.open_questions: list[str] = []
        self.decisions: list[str] = []

    def confirm(self, key: str, finding: str) -> None:
        if not EVIDENCE.search(finding):
            raise ValueError(f"no evidence (file:line or tool_result:<id>) in {finding!r}")
        self.findings[key] = finding  # same key = superseded entry is rewritten, not appended

    def render(self) -> str:
        parts = ["# investigation-scratchpad.md   (agent-maintained; read this before answering)", "",
                 "## Confirmed findings (each with evidence)"]
        parts += [f"- [{k}] {v}" for k, v in self.findings.items()]
        parts += ["", "## Open questions"] + [f"- {q}" for q in self.open_questions]
        parts += ["", "## Decisions"] + [f"- {d}" for d in self.decisions]
        return "\n".join(parts) + "\n"

    def save(self) -> None:
        self.path.write_text(self.render())

    @classmethod
    def load(cls, path: Path) -> "Scratchpad":
        pad, section = cls(path), None
        for line in path.read_text().splitlines():
            if line.startswith("## "):
                section = line[3:].split(" (")[0]
            elif line.startswith("- "):
                item = line[2:]
                if section == "Confirmed findings":
                    key, _, finding = item.partition("] ")
                    pad.findings[key.lstrip("[")] = finding
                elif section == "Open questions":
                    pad.open_questions.append(item)
                elif section == "Decisions":
                    pad.decisions.append(item)
        return pad


pad = Scratchpad(WORKDIR / "investigation-scratchpad.md")
for key, finding in [
    ("refund-service", "RefundService.issue() (src/billing/refunds.ts:41) needs manager approval over $50"),
    ("refund-callers", "issue() is called from SupportController (src/api/support.ts:57) and RefundRetryJob (src/jobs/retry.ts:22)"),
    ("vague", "payments is complicated"),
    ("rate-limit", "PaymentGateway rate limit: 100 req/min (src/payments/gateway.ts:7)"),
]:
    try:
        pad.confirm(key, finding)
        print("✓ recorded:", key)
    except ValueError as e:
        print("✗ rejected:", e)
pad.confirm("rate-limit", "PaymentGateway rate limit: 120 req/min after config change (src/payments/gateway.ts:9)")  # supersede
pad.open_questions.append("Does RefundRetryJob back off on gateway 429s? (not yet read)")
pad.decisions.append("Scope: card refunds only; wallet refunds out of scope (agreed turn 14)")
pad.save()
print("\n" + pad.path.read_text())

Next morning, a fresh session starts from the file, not from yesterday's transcript:

In [ ]:
restored = Scratchpad.load(WORKDIR / "investigation-scratchpad.md")
assert restored.findings == pad.findings and restored.decisions == pad.decisions
first_message = (
    "<scratchpad>\n" + restored.render() + "</scratchpad>\n"
    "Before answering a question about the codebase, consult the scratchpad above. "
    "After you confirm a new fact, add it with its file and line.\n\n"
    "Question: where would a retry for gateway 429s have to go?"
)
print(f"new session starts with ~{est_tokens(first_message)} tokens of distilled findings")

> **Observe:** the vague entry was rejected, the rate-limit entry was rewritten instead of duplicated, and the next session gets a couple of hundred tokens of specifics instead of hours of transcript. In Claude Code, the same order matters: write critical findings to the file **first**, then `/compact`.

## 11.5 Delegate exploration to subagents to protect the coordinator's context

Exploration is verbose and its answer is short. A subagent reads in its **own fresh context window** and returns only its final message, so the coordinator keeps a few hundred tokens instead of tens of thousands. Subagents do not inherit the coordinator's history: pass the minimal facts explicitly, restrict tools to the role, ask for structured results, and inject a phase-1 summary into phase-2 prompts.

[Theory](README.md#115-delegate-exploration-to-subagents-to-protect-the-coordinators-context)

In [ ]:
# A fake 20-file repo: every file is long, and only its import lines matter for the question.
REPO = {
    f"src/payments/mod{i}.ts": (f"import {{ Dep{i} }} from '../lib/dep{i}';\n" if i % 3 == 0 else "")
    + (f"import {{ AuthService }} from '../auth/service';\n" if i == 4 else "")
    + "// implementation details\n" + "export function fn() { return compute(); }\n" * 120
    for i in range(20)
}


def dependency_tracer(question: str, files: dict[str, str]) -> dict:
    """Stand-in for a read-only subagent: reads everything in ITS window, returns only a structured result."""
    deps = []
    for path, src in files.items():
        for line_no, line in enumerate(src.splitlines(), 1):
            m = re.match(r"import \{ (\w+) \} from '([^']+)'", line)
            if m:
                deps.append({"name": m.group(1), "file": path, "line": line_no, "kind": "import"})
    return {"dependencies": deps, "open_questions": []}


read_cost = sum(est_tokens(src) for src in REPO.values())
result = dependency_tracer("Trace the payments module's dependencies.", REPO)
print(f"coordinator reads the files itself: ~{read_cost:,} tokens stay in its context for the rest of the session")
print(f"subagent does it: the coordinator keeps ~{est_tokens(result)} tokens -> {json.dumps(result)[:160]}...")

### Minimal, explicit subagent prompts (offline)

The subagent cannot see the coordinator's conversation. Its prompt carries the question, the specific facts it needs and, for a phase-2 task, the distilled phase-1 findings, and never the coordinator's transcript.

In [ ]:
def build_subagent_prompt(question: str, facts_needed: list[str], phase_summary: str | None = None,
                          output_contract: str = '{"findings": [{"claim", "file", "line"}], "gaps": [...]}') -> str:
    parts = [f"<task>\n{question}\n</task>"]
    if facts_needed:
        parts.append("<facts>\n" + "\n".join(f"- {f}" for f in facts_needed) + "\n</facts>")
    if phase_summary:
        parts.append(f"<phase1_findings>\n{phase_summary}\n</phase1_findings>")
    parts.append(f"Return ONLY JSON matching {output_contract}. No prose, no file contents.")
    return "\n\n".join(parts)


coordinator_transcript = "\n".join(f"turn {i}: ... long exploration output ..." * 20 for i in range(40))
prompt = build_subagent_prompt(
    "Find every place that must change to add a `reason` argument to RefundService.issue().",
    facts_needed=["RefundService.issue() is defined in src/billing/refunds.ts:41"],
    phase_summary=Scratchpad.load(WORKDIR / "investigation-scratchpad.md").render(),
)
assert "long exploration output" not in prompt
print(f"subagent prompt ~{est_tokens(prompt)} tokens (coordinator transcript would have been ~{est_tokens(coordinator_transcript):,})")
print(prompt[:500])

The subagent definition mirrors the SDK's `AgentDefinition` fields. Note the two different "tool lists": a subagent's own toolset is `AgentDefinition(tools=[...])`, while `ClaudeAgentOptions(allowed_tools=[...])` lists tools that run without a permission prompt. The guide (and probably the exam) calls the spawning tool `Task`; current versions emit `Agent`, so match both.

In [ ]:
AGENTS = {  # same fields as claude_agent_sdk.AgentDefinition (description, prompt, tools, model)
    "dependency-tracer": {
        "description": "Traces which modules a given module depends on. Read-only.",
        "prompt": "Answer ONLY the question you are given. Return just JSON: {dependencies, open_questions}.",
        "tools": ["Read", "Grep", "Glob"],          # role-scoped toolset
        "model": "claude-haiku-5-5",                # cheap model for bulk reading
    },
}
SUBAGENT_TOOL_NAMES = {"Agent", "Task"}  # current name and legacy name


def is_subagent_call(block: dict) -> bool:
    return block.get("type") == "tool_use" and block.get("name") in SUBAGENT_TOOL_NAMES


for b in [tool_use("Agent", {"subagent_type": "dependency-tracer"}), tool_use("Task", {}), tool_use("Read", {})]:
    print(b["name"], "->", is_subagent_call(b))
print("write tools in the read-only tracer:", set(AGENTS["dependency-tracer"]["tools"]) & {"Write", "Edit", "Bash"})

In a real coordinator (not run here):

```python
from claude_agent_sdk import ClaudeAgentOptions, AgentDefinition

options = ClaudeAgentOptions(
    model=MODEL,
    agents={name: AgentDefinition(**spec) for name, spec in AGENTS.items()},
    allowed_tools=["Agent", "Read"],   # coordinator: delegate, and read the scratchpad
    system_prompt="You coordinate a codebase investigation. Delegate file-heavy questions to subagents. ...",
)
```

In Claude Code, the built-in read-only **Explore** subagent and skills with `context: fork` give you the same isolation.

## 11.6 Structured state persistence and crash recovery

Each agent writes its structured state (status, key findings, coverage, gaps) to a known file **after every status change**, the coordinator keeps a **manifest**, and on restart it skips completed agents, re-runs in-progress or failed ones (so reruns must be idempotent), and injects completed findings into the remaining prompts. Writes are **atomic**: write a temp file, then rename.

[Theory](README.md#116-structured-state-persistence-and-crash-recovery)

In [ ]:
def write_json_atomic(path: Path, data: dict) -> None:
    """Write to a temp file, then rename: a crash never leaves a half-written state file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, suffix=".tmp")
    try:
        with os.fdopen(fd, "w") as f:
            json.dump(data, f, indent=2)
        os.replace(tmp, path)                     # atomic on POSIX and Windows
    except BaseException:
        os.unlink(tmp)                            # never leave temp files behind
        raise


def write_json_naive(path: Path, data: dict) -> None:
    with open(path, "w") as f:
        json.dump(data, f, indent=2)


class Unserializable:  # stands in for a crash halfway through json.dump
    pass


for writer in (write_json_naive, write_json_atomic):
    p = WORKDIR / f"state-{writer.__name__}.json"
    write_json_atomic(p, {"status": "completed", "key_findings": ["a", "b"]})
    try:
        writer(p, {"status": "in_progress", "key_findings": ["a", "b", Unserializable()]})
    except TypeError:
        pass
    try:
        print(f"{writer.__name__:<18} after a failed write the file still parses:", json.loads(p.read_text())["status"])
    except json.JSONDecodeError as e:
        print(f"{writer.__name__:<18} after a failed write the file is CORRUPT: {e}")

### A coordinator that survives a crash (offline)

In [ ]:
STATE_DIR = WORKDIR / "agent-state"
RUNS = {"web-search": 0, "doc-analysis": 0, "synthesis": 0}   # how often each agent actually ran


class Crash(Exception):
    pass


def web_search(prior: str) -> dict:
    RUNS["web-search"] += 1
    return {"status": "completed", "searches": ["studio AI adoption survey"], "sources_read": 9,
            "key_findings": [{"claim": "AI composition use grew to 31%", "source": "https://example.org/a", "date": "2024-11-02"}],
            "coverage": ["studio adoption"], "gaps": ["licensing revenue"]}


def doc_analysis(prior: str) -> dict:
    RUNS["doc-analysis"] += 1
    assert "AI composition use grew" in prior, "completed findings must be injected into this agent's prompt"
    return {"status": "completed", "key_findings": [{"claim": "Licensing revenue fell 8%", "source": "report.pdf p.4", "date": "2024-09-30"}],
            "coverage": ["licensing revenue"], "gaps": []}


def synthesis(prior: str) -> dict:
    RUNS["synthesis"] += 1
    return {"status": "completed", "key_findings": [{"claim": "Report drafted from 2 agents", "source": "synthesis", "date": "2026-10-03"}],
            "coverage": ["all"], "gaps": []}


PIPELINE = {"web-search": web_search, "doc-analysis": doc_analysis, "synthesis": synthesis}


def resume_plan(manifest: dict) -> tuple[list[str], str]:
    """Which agents still need to run, and the findings to inject into their prompts."""
    done, todo = [], []
    for name, entry in manifest["agents"].items():
        (done if entry["status"] == "completed" else todo).append(name)
    injected = []
    for name in done:
        state = json.loads(Path(manifest["agents"][name]["state_file"]).read_text())
        injected.append(f"<prior_results agent='{name}'>\n{json.dumps(state['key_findings'])}\n"
                        f"coverage={state['coverage']} gaps={state['gaps']}\n</prior_results>")
    return todo, "\n".join(injected)


def run_coordinator(crash_before: str | None = None) -> None:
    mpath = STATE_DIR / "manifest.json"
    manifest = json.loads(mpath.read_text()) if mpath.exists() else {
        "run_id": "research-2026-10-03-a", "schema_version": 2,
        "agents": {n: {"status": "not_started", "state_file": str(STATE_DIR / f"{n}.json")} for n in PIPELINE}}
    todo, _ = resume_plan(manifest)
    print("to run:", todo)
    for name in todo:
        entry = manifest["agents"][name]
        entry["status"], entry["attempt"] = "in_progress", entry.get("attempt", 0) + 1
        write_json_atomic(mpath, manifest)                     # state after EVERY status change
        if name == crash_before:
            raise Crash(f"process died while {name} was running")
        _, prior = resume_plan(manifest)                       # inject what is completed so far
        state = PIPELINE[name](prior)
        write_json_atomic(Path(entry["state_file"]), state)    # agent state first...
        entry["status"] = "completed"
        write_json_atomic(mpath, manifest)                     # ...then the manifest


try:
    run_coordinator(crash_before="doc-analysis")
except Crash as e:
    print("CRASH:", e)
print("manifest on disk:", {n: e["status"] for n, e in json.loads((STATE_DIR / "manifest.json").read_text())["agents"].items()})
run_coordinator()  # restart: loads the manifest, skips web-search, re-runs doc-analysis with web-search's findings
print("runs per agent:", RUNS)

> **Observe:** `web-search` ran once, even though the process crashed. `doc-analysis` ran with web-search's findings in its prompt (its `assert` checks that). The manifest is written after every status change, so the restart knows exactly where it stopped. Write the agent's state file **before** marking it completed in the manifest: a crash between the two writes then causes one idempotent rerun, never a "completed" entry with no state file.

> **Try it:** add `crash_before="synthesis"` and restart twice. How many times does each agent run?

## 11.7 Sessions: resume, fork, or start fresh (task 1.7)

What you resume is the context, including every old tool result in it. Resume when prior context is still valid; resume **and list the changed files** when a few known files changed; **fork** to explore two approaches from one baseline; start **fresh with a structured summary** when many or unknown things changed, because stale tool results produce confident wrong answers.

[Theory](README.md#117-sessions-resume-fork-or-start-fresh-task-17)

In [ ]:
def choose_session(changed_files: list[str] | None, compare_approaches: bool = False, few: int = 3) -> str:
    """changed_files: None = unknown; [] = nothing changed."""
    if compare_approaches:
        return "fork"
    if changed_files is None or len(changed_files) > few:
        return "fresh_with_summary"
    return "resume_with_changes" if changed_files else "resume"


def change_notice(changed: dict[str, str]) -> str:
    lines = ["Since this session last ran, these files changed:"]
    lines += [f"- {path} ({what})" for path, what in changed.items()]
    lines.append("Your earlier reads of these files are outdated. Re-read them before answering, then update "
                 "the dependency map. Everything else you analyzed is unchanged.")
    return "\n".join(lines)


def fresh_start_summary(pad: Scratchpad, task: str) -> str:
    return (f"<previous_investigation>\n{pad.render()}</previous_investigation>\n"
            "Many files may have changed since then: treat findings as leads and re-read any file before relying on it.\n"
            f"Task: {task}")


for scenario, args in [("nothing changed", ([],)), ("2 known files changed", (["processor.ts", "gateway.ts"],)),
                       ("big merge, unknown changes", (None,)), ("compare 2 test strategies", ([], True))]:
    print(f"{scenario:<28} -> {choose_session(*args)}")
print()
print(change_notice({"src/payments/processor.ts": "refund() now takes a reason argument",
                     "src/payments/gateway.ts": "retry logic added"}))

The matching mechanics (see [Chapter 5](../05-claude-code/README.md) and [Chapter 3](../03-agent-sdk/README.md)):

| Decision | Claude Code CLI | Agent SDK (Python) |
|---|---|---|
| `resume` / `resume_with_changes` | `claude --resume auth-bug` (named earlier with `claude -n auth-bug`), then paste `change_notice(...)` | `ClaudeAgentOptions(resume=session_id)` |
| `fork` | `claude --resume <id> --fork-session`, or `/branch` | `ClaudeAgentOptions(resume=session_id, fork_session=True)` |
| `fresh_with_summary` | `/clear` or a new session; first message = `fresh_start_summary(...)` | a new `query()` whose prompt is the summary |

### Session hygiene: a session linter (offline)

Session hygiene is the day-to-day version of the same decision: keep each session about one task, and clean up before the context works against you. The rules below come from the Claude Code best-practices guide ("kitchen sink" sessions, repeated corrections, unscoped exploration) plus the application-side rules from the README. The linter takes a few facts about a session and returns the hygiene actions it needs.

[Theory](README.md#session-hygiene)

In [ ]:
def session_hygiene(task_topics: list[str], corrections_on_same_issue: int, context_used: float,
                    files_read_in_main: int, next_task_related: bool, named: bool) -> list[str]:
    """Return hygiene actions for one interactive session (Claude Code or a chat app)."""
    actions = []
    if len(set(task_topics)) > 1:
        actions.append("kitchen sink: one task per session; /clear (or a new session) between unrelated tasks")
    if corrections_on_same_issue >= 2:   # two failed corrections: restart instead of a third
        actions.append("context is polluted with failed attempts: /clear and restart with a better prompt "
                       "that includes what you learned")
    if files_read_in_main > 20:
        actions.append("unscoped exploration in the main context: scope it, or delegate to a subagent")
    if context_used > 0.6 and next_task_related:
        actions.append("write key findings to a scratchpad, then /compact <focus> (or /rewind -> summarize)")
    if not next_task_related:
        actions.append("next task is unrelated: /clear instead of compacting")
    if not named:
        actions.append("name the session (claude -n <name> or /rename) so it can be resumed or forked later")
    return actions or ["healthy: keep going"]


cases = {
    "focused, half full": (["refund-bug"], 0, 0.45, 6, True, True),
    "kitchen sink": (["refund-bug", "css-question", "refund-bug"], 0, 0.55, 8, True, True),
    "three failed fixes": (["refund-bug"], 3, 0.70, 12, True, False),
    "big exploration, switching task": (["map-billing"], 0, 0.80, 45, False, True),
}
for name, args in cases.items():
    print(f"{name}:")
    for a in session_hygiene(*args):
        print("   -", a)
assert session_hygiene(["a"], 0, 0.3, 2, True, True) == ["healthy: keep going"]
assert any("/clear" in a for a in session_hygiene(["a", "b"], 0, 0.3, 2, True, True))

> **Observe:** none of these actions needs a bigger window. In an API application the same hygiene is structural: one conversation per case (never mix two customers' histories), session state scoped to a user or tenant with an expiry, and a fresh conversation seeded with the case-facts block when a case is reopened.

## 11.8 Context windows, token budgets and counting *(beyond the guide)*

Fable 5.1, Opus 5.5, Sonnet 5.5 and Haiku 5.5 have a 1M-token window; legacy Haiku 4.5 has 200K. Everything counts: system, tools, every message, and the current output including thinking. Treat the window as a ceiling and budget it by layer. **Count before** a request with the free `count_tokens` endpoint; **measure after** from `usage`, where cached input is split across three fields that all count toward the window.

[Theory](README.md#118-context-windows-token-budgets-and-counting-beyond-the-guide)

In [ ]:
BUDGET = {  # the support manager's working budget (tokens)
    "system + tools (cached)": 4_000,
    "case facts": 1_000,
    "summary of older turns": 2_000,
    "recent turns verbatim": 12_000,
    "trimmed tool results in flight": 6_000,
    "output reserve (max_tokens)": 16_000,
}
INPUT_BUDGET = sum(v for k, v in BUDGET.items() if not k.startswith("output"))
print(f"input budget {INPUT_BUDGET:,} + output reserve {BUDGET['output reserve (max_tokens)']:,} "
      f"= {sum(BUDGET.values()):,} of a 1,000,000-token window")


def window_used(u) -> int:
    """Everything that occupied the window on one request. The cache fields are Optional in the SDK."""
    return (u.input_tokens + (u.cache_read_input_tokens or 0)
            + (u.cache_creation_input_tokens or 0) + u.output_tokens)


usage_examples = {
    "no caching": anthropic.types.Usage(input_tokens=18_000, output_tokens=600),
    "warm cache": anthropic.types.Usage(input_tokens=900, cache_read_input_tokens=17_000,
                                        cache_creation_input_tokens=100, output_tokens=600),
}
for name, u in usage_examples.items():
    print(f"{name:<11} input_tokens={u.input_tokens:>6,}  window used={window_used(u):,}")

> **Observe:** with a warm cache, `input_tokens` alone is tiny, but the window holds just as much. Budget checks that read only `input_tokens` silently overflow.

### Count before you send (offline shape, then live)

In [ ]:
SUPPORT_SYSTEM = "You are a support agent. Follow the refund policy exactly."
TOOLS = [{"name": "lookup_order", "description": "Look up an order by ID. Returns status, total, dates and items.",
          "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}}]
history = [{"role": "user", "content": "Where is my refund for ORD-30417?"}]

fake = ScriptedClaude()
n = fake.client.messages.count_tokens(model=MODEL, system=SUPPORT_SYSTEM, tools=TOOLS, messages=history)
print("offline count (estimate from the fake):", n.input_tokens, "| request went to", fake.requests[-1]["path"])


In [ ]:
# (live, free) the real count with the model you will call; newer tokenizers count differently, so never reuse old numbers
real = client.messages.count_tokens(model=MODEL, system=SUPPORT_SYSTEM, tools=TOOLS, messages=history)
print("count_tokens:", real.input_tokens, "| offline estimate:", est_tokens([SUPPORT_SYSTEM, TOOLS, history]))

## 11.9 Server-side compaction *(beta, beyond the guide)*

Compaction replaces older turns with a summary Claude writes on the server. **On demand** (`compaction={"type": "summarize"}`, beta `compact-2026-09-04`) returns only a signed `compaction` block with `stop_reason: "compaction"`; you put that block **first** in place of the messages it summarized. **At a threshold** (`context_management` edit `compact_20260112`, beta `compact-2026-01-12`) the API compacts inside a normal reply. It is still summarization, so the case-facts block stays outside it.

[Theory](README.md#119-server-side-compaction-beyond-the-guide)

In [ ]:
COMPACT_AT = 120_000          # your input budget, below the model's window
COMPACT_INSTRUCTIONS = ("Summarize this support conversation so it can continue. Preserve every order ID, amount, "
                        "percentage, date and customer-stated expectation exactly as written. Do not call tools; "
                        "respond with the summary only.")


def maybe_compact(cl: anthropic.Anthropic, history: list, system: str, tools: list, last_usage) -> list:
    used = (last_usage.input_tokens + (last_usage.cache_read_input_tokens or 0)
            + (last_usage.cache_creation_input_tokens or 0) + last_usage.output_tokens)
    if used < COMPACT_AT:
        return history
    summary = cl.beta.messages.create(
        model=MODEL, max_tokens=8192, system=system, tools=tools, betas=["compact-2026-09-04"],  # same system + tools
        messages=history, compaction={"type": "summarize", "instructions": COMPACT_INSTRUCTIONS},
    )
    if summary.stop_reason != "compaction":       # max_tokens, refusal, tool_use...: keep history, retry later
        return history
    return [{"role": "assistant", "content": summary.content}]   # the block replaces what it summarized


# 20 answered exchanges, ending on an assistant reply (compact after a reply, so no question is left unanswered)
long_history = [{"role": "user" if i % 2 == 0 else "assistant", "content": f"turn {i} ..."} for i in range(40)]
big_usage = anthropic.types.Usage(input_tokens=2_000, cache_read_input_tokens=125_000, output_tokens=500)

fake = ScriptedClaude(reply({"type": "compaction", "content": "Summary: refund of 74.95 USD for ORD-30417 pending ...",
                             "signature": "EuYBCkQY-scripted"}, stop_reason="compaction",
                            usage={"input_tokens": 0, "output_tokens": 0,
                                   "iterations": [{"type": "compaction", "input_tokens": 131_000, "output_tokens": 900,
                                                   "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0}]}))
new_history = maybe_compact(fake.client, long_history, SUPPORT_SYSTEM, TOOLS, big_usage)
sent = fake.requests[-1]
print("beta header:", sent["beta"], "| compaction param:", sent["body"]["compaction"]["type"])
print(f"history: {len(long_history)} messages -> {len(new_history)}; first block type:", new_history[0]["content"][0].type)

The next request sends the block **exactly as returned** (signature included), first in `messages`, with the beta header. Two checks you can run before sending:

In [ ]:
def compaction_errors(kwargs: dict) -> list[str]:
    """Pre-flight checks for requests that use compaction or context management (documented rules)."""
    errs, betas = [], set(kwargs.get("betas", []))
    if "compaction" in kwargs and "context_management" in kwargs:
        errs.append("compaction and context_management cannot be combined on one request")
    if "compaction" in kwargs:
        if "compact-2026-09-04" not in betas:
            errs.append("on-demand compaction needs the compact-2026-09-04 beta header")
        if len(kwargs["compaction"].get("instructions") or "") > 16_384:
            errs.append("compaction instructions are limited to 16,384 characters")
    for edit in kwargs.get("context_management", {}).get("edits", []):
        if edit["type"] == "compact_20260112":
            if "compact-2026-01-12" not in betas:
                errs.append("threshold compaction needs the compact-2026-01-12 beta header")
            if edit.get("trigger", {}).get("value", 150_000) < 50_000:
                errs.append("compact_20260112 trigger must be at least 50,000 input tokens")
    for i, m in enumerate(kwargs.get("messages", [])):
        blocks = m["content"] if isinstance(m["content"], list) else []
        if i > 0 and any((b.get("type") if isinstance(b, dict) else b.type) == "compaction" for b in blocks):
            errs.append(f"compaction block at messages[{i}]: summarized messages must be removed (400 compaction_block_misplaced)")
    return errs


ok_next = {"betas": ["compact-2026-09-04"], "messages": new_history + [{"role": "user", "content": "Any update?"}]}
appended = {"betas": ["compact-2026-09-04"], "messages": long_history + new_history + [{"role": "user", "content": "Any update?"}]}
threshold = {"betas": ["compact-2026-01-12"], "context_management": {"edits": [
    {"type": "compact_20260112", "trigger": {"type": "input_tokens", "value": 30_000}}]}}
both = {"betas": ["compact-2026-09-04"], "compaction": {"type": "summarize"},
        "context_management": {"edits": [{"type": "clear_tool_uses_20250919"}]}}
for name, kw in [("block first", ok_next), ("block appended", appended), ("trigger 30K", threshold), ("both params", both)]:
    print(f"{name:<15}", compaction_errors(kw) or "ok")

In [ ]:
# Where the cost of a compaction call is reported
fake = ScriptedClaude(reply({"type": "compaction", "content": "Summary ...", "signature": "sig-scripted"}, stop_reason="compaction",
                            usage={"input_tokens": 0, "output_tokens": 0, "iterations": [
                                {"type": "compaction", "input_tokens": 131_000, "output_tokens": 900,
                                 "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0}]}),
                      reply(stop_reason="max_tokens"))   # an unfinished compaction call: 200 with empty content
r = fake.client.beta.messages.create(model=MODEL, max_tokens=8192, betas=["compact-2026-09-04"],
                                     messages=long_history, compaction={"type": "summarize"})
print("top-level usage:", r.usage.input_tokens, "in /", r.usage.output_tokens, "out")
print("usage.iterations:", [(it.type, it.input_tokens, it.output_tokens) for it in r.usage.iterations])

# A summarization call that did not finish: maybe_compact keeps the history unchanged
kept = maybe_compact(fake.client, long_history, SUPPORT_SYSTEM, TOOLS, big_usage)
print("stop_reason max_tokens -> history kept:", kept is long_history)

> **Observe:** the top-level `input_tokens` / `output_tokens` of a compaction response are 0; the real cost is reported in `usage.iterations`. If the summarization call does not end normally (`stop_reason` is not `"compaction"`), `maybe_compact` keeps the history and tries again later.

**(live)** Compact a short real conversation on demand, then continue from the block.

In [ ]:
convo = [
    {"role": "user", "content": "My blender (order ORD-30417, $74.95, ordered 2025-01-15) arrived damaged. I want a full refund, not store credit."},
    {"role": "assistant", "content": "I'm sorry. I've opened a refund request for ORD-30417 and sent it for manager approval."},
    {"role": "user", "content": "Also, your colleague promised me 25% off my next order."},
    {"role": "assistant", "content": "Noted. I'll check that promise with the team."},
]
summary = client.beta.messages.create(model=MODEL, max_tokens=4096, betas=["compact-2026-09-04"], messages=convo,
                                      compaction={"type": "summarize", "instructions": COMPACT_INSTRUCTIONS})
print("stop_reason:", summary.stop_reason, "| iterations:", summary.usage.iterations)
if summary.stop_reason == "compaction":
    block = summary.content[0]
    print("summary keeps:", surviving(["ORD-30417", "74.95", "2025-01-15", "store credit", "25%"], block.content or ""))
    nxt = client.beta.messages.create(model=MODEL, max_tokens=300, betas=["compact-2026-09-04"], messages=[
        {"role": "assistant", "content": summary.content},
        {"role": "user", "content": "What exactly did I ask for? One sentence."}])
    print("continued:", text_of(nxt).strip())

## 11.10 Context editing: clearing tool results and thinking *(beta, beyond the guide)*

Context editing clears content **by rule, on the server**, before the prompt reaches Claude, while your client keeps the full history. `clear_tool_uses_20250919` replaces the oldest tool results with a placeholder once the prompt passes a trigger (default 100,000 input tokens, keep the last 3 tool uses); `clear_thinking_20251015` clears earlier thinking and must come first when combined. Clearing breaks the cache from the cleared point, so set `clear_at_least` to make each break worth it.

[Theory](README.md#1110-context-editing-clearing-tool-results-and-thinking-beyond-the-guide)

In [ ]:
def clearing_edits(trigger_tokens: int = 60_000, keep: int = 4, clear_at_least: int = 10_000,
                   exclude: list[str] | None = None, clear_thinking_keep: int | None = None) -> dict:
    edits = []
    if clear_thinking_keep is not None:  # must be listed first when combined
        edits.append({"type": "clear_thinking_20251015", "keep": {"type": "thinking_turns", "value": clear_thinking_keep}})
    edits.append({
        "type": "clear_tool_uses_20250919",
        "trigger": {"type": "input_tokens", "value": trigger_tokens},
        "keep": {"type": "tool_uses", "value": keep},
        "clear_at_least": {"type": "input_tokens", "value": clear_at_least},  # make each cache break worth it
        "exclude_tools": exclude or [],
    })
    return {"edits": edits}


cm = clearing_edits(exclude=["get_customer"], clear_thinking_keep=2)
assert cm["edits"][0]["type"] == "clear_thinking_20251015"

fake = ScriptedClaude(reply(text("Refund approved."), context_management={"applied_edits": [
    {"type": "clear_tool_uses_20250919", "cleared_tool_uses": 8, "cleared_input_tokens": 50_000}]}))
r = fake.client.beta.messages.create(model=MODEL, max_tokens=1024, betas=["context-management-2025-06-27"],
                                     tools=TOOLS, messages=history, context_management=cm)
print("beta:", fake.requests[-1]["beta"], "| edits sent:", [e["type"] for e in fake.requests[-1]["body"]["context_management"]["edits"]])
print("applied_edits:", r.context_management.applied_edits)

preview = fake.client.beta.messages.count_tokens(model=MODEL, betas=["context-management-2025-06-27"],
                                                 tools=TOOLS, messages=history, context_management=cm)
print("count_tokens preview -> original:", preview.context_management.original_input_tokens, "after edits:", preview.input_tokens)

### What the model sees vs what you keep (offline illustration)

The server's placeholder text is its own; the local simulation below only shows the *shape* of the result: old results are gone from the model's view, the newest `keep` results and excluded tools stay, tool calls stay visible (`clear_tool_inputs` defaults to `false`), and your `history` list is untouched.

In [ ]:
def simulate_clear_tool_uses(messages: list[dict], keep: int, exclude: set[str]) -> list[dict]:
    names = {b["id"]: b["name"] for m in messages if isinstance(m["content"], list)
             for b in m["content"] if b.get("type") == "tool_use"}
    result_ids = [b["tool_use_id"] for m in messages if isinstance(m["content"], list)
                  for b in m["content"] if b.get("type") == "tool_result"]
    clearable = [i for i in result_ids if names.get(i) not in exclude]
    to_clear = set(clearable[:-keep] if keep else clearable)
    view = copy.deepcopy(messages)
    for m in view:
        for b in (m["content"] if isinstance(m["content"], list) else []):
            if b.get("type") == "tool_result" and b["tool_use_id"] in to_clear:
                b["content"] = "[cleared]"
    return view


loop = [{"role": "user", "content": "Investigate the refund."}]
for i, name in enumerate(["get_customer", "lookup_order", "lookup_order", "read_log", "read_log", "read_log"]):
    loop.append({"role": "assistant", "content": [tool_use(name, {}, id=f"t{i}")]})
    loop.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": f"t{i}", "content": f"{name} output " * 200}]})
view = simulate_clear_tool_uses(loop, keep=2, exclude={"get_customer"})
for m_full, m_view in zip(loop, view):
    for b_full, b_view in zip(m_full["content"] if isinstance(m_full["content"], list) else [],
                              m_view["content"] if isinstance(m_view["content"], list) else []):
        if b_full.get("type") == "tool_result":
            print(f"{b_full['tool_use_id']}: client keeps ~{est_tokens(b_full['content'])} tokens | model sees ~{est_tokens(b_view['content'])}")

**(live, free)** Preview the saving of a clearing rule with the real token-counting endpoint.

In [ ]:
real_tools = TOOLS + [{"name": "read_log", "description": "Read a log file.",
                       "input_schema": {"type": "object", "properties": {}}},
                      {"name": "get_customer", "description": "Get a customer.", "input_schema": {"type": "object", "properties": {}}}]
preview = client.beta.messages.count_tokens(
    model=MODEL, betas=["context-management-2025-06-27"], tools=real_tools,
    messages=loop + [{"role": "assistant", "content": "Done reading."}, {"role": "user", "content": "Summarize."}],
    context_management={"edits": [{"type": "clear_tool_uses_20250919", "trigger": {"type": "tool_uses", "value": 3},
                                   "keep": {"type": "tool_uses", "value": 2}, "exclude_tools": ["get_customer"]}]},
)
print("original:", preview.context_management.original_input_tokens, "-> after clearing:", preview.input_tokens)

## 11.11 The memory tool *(beyond the guide)*

The memory tool (`{"type": "memory_20250818", "name": "memory"}`, generally available) is the API's built-in scratchpad: Claude issues `view`, `create`, `str_replace`, `insert`, `delete` and `rename` commands against a `/memories` directory, and **your application executes them** against storage you control. That makes path validation, size caps and per-tenant scoping your job. The Python SDK ships `BetaLocalFilesystemMemoryTool` (and `BetaAbstractMemoryTool` to subclass for your own storage).

[Theory](README.md#1111-the-memory-tool-beyond-the-guide)

In [ ]:
from anthropic.tools import BetaLocalFilesystemMemoryTool

memory = BetaLocalFilesystemMemoryTool(base_path=str(WORKDIR / "memory" / "case-CUST-58214"))  # one directory per customer
print("tool definition:", memory.to_dict())
print(memory.call({"command": "create", "path": "/memories/case.md",
                   "file_text": "# CUST-58214\n- ORD-30417: refund 74.95 USD pending approval\n"}))
print(memory.call({"command": "str_replace", "path": "/memories/case.md",
                   "old_str": "pending approval", "new_str": "approved 2026-10-03"}))
print(memory.call({"command": "view", "path": "/memories/case.md"}))

for attack in ["/memories/../../other-customer/case.md", "/etc/passwd", "../secrets"]:
    try:
        memory.call({"command": "view", "path": attack})
    except Exception as e:
        print(f"blocked {attack!r}: {type(e).__name__}: {e}")

### Your own storage needs the same guard (offline)

If you back memory with a database or blob store (subclass `BetaAbstractMemoryTool`), you write the path check yourself. Decode URL-encoding, resolve the path, and require it to stay inside the tenant's root:

In [ ]:
def safe_memory_path(tenant_root: Path, requested: str, max_decode: int = 3) -> Path:
    """Map a /memories/... path onto tenant_root, or raise ValueError."""
    decoded = requested
    for _ in range(max_decode):  # catch double-encoded traversal like %252e%252e
        nxt = urllib.parse.unquote(decoded)
        if nxt == decoded:
            break
        decoded = nxt
    if decoded != "/memories" and not decoded.startswith("/memories/"):
        raise ValueError(f"path must start with /memories: {requested!r}")
    root = tenant_root.resolve()
    target = (root / decoded.removeprefix("/memories").lstrip("/")).resolve()
    if not target.is_relative_to(root):
        raise ValueError(f"path escapes the memory root: {requested!r}")
    return target


root = WORKDIR / "tenant-a"
for p in ["/memories/notes.md", "/memories/sub/../notes.md", "/memories/../tenant-b/x.md",
          "/memories/%2e%2e/tenant-b/x.md", "/memories/%252e%252e/x", "/etc/passwd"]:
    try:
        print(f"ok      {p:<32} -> {safe_memory_path(root, p).relative_to(WORKDIR.resolve())}")
    except ValueError as e:
        print(f"blocked {p:<32} ({e})")

### The tool loop, offline (the SDK's tool runner with a scripted model)

`client.beta.messages.tool_runner(...)` runs the loop: it executes each memory command with your handler and sends the result back. With `ScriptedClaude` you can watch the whole exchange without a key.

In [ ]:
fake = ScriptedClaude(
    reply(tool_use("memory", {"command": "view", "path": "/memories"})),
    reply(tool_use("memory", {"command": "view", "path": "/memories/case.md"})),
    reply(text("Your refund of 74.95 USD for ORD-30417 was approved on 2026-10-03.")),
)
runner = fake.client.beta.messages.tool_runner(model=MODEL, max_tokens=1024, tools=[memory], messages=[
    {"role": "user", "content": "Continue case CUST-58214. Check your notes first. What is the refund status?"}])
final = runner.until_done()
print("final:", text_of(final))
print("requests:", len(fake.requests), "| tool sent:", fake.requests[0]["body"]["tools"])
print("2nd request's last message (the view result your handler produced):")
print(json.dumps(fake.requests[2]["body"]["messages"][-1])[:300])

In [ ]:
# (live) the same loop against the real model. Claude decides what to read and write in /memories.
runner = client.beta.messages.tool_runner(model=MODEL, max_tokens=1024, tools=[memory], messages=[
    {"role": "user", "content": "Continue case CUST-58214. Check your notes first, then tell me the refund status in one sentence."}])
final = runner.until_done()
print(final.stop_reason, "|", text_of(final).strip())

> **Observe:** the model only *asks* for memory operations; your handler performs them. Treat what comes back from memory as helpful notes to verify, not as authoritative facts (those live in the case-facts block and your systems of record).

## 11.12 Prompt caching and context management *(beyond the guide)*

Caching changes what you **pay** for tokens, not whether they count toward the window. The cache key is the exact prefix in the order `tools` → `system` → `messages`, so static content goes first and changing state (case facts, timestamps) goes last. Cache writes cost 1.25× base input (5-minute TTL) and reads 0.1× on most models (0.05× on Opus 5.5 and Sonnet 5.5, 0.025× on Fable 5.1); the minimum cacheable prefix is 512 tokens on current models and 4,096 on legacy Haiku 4.5.

[Theory](README.md#1112-prompt-caching-and-context-management-beyond-the-guide)

In [ ]:
PRICE = {"input": 2.00, "write_5m": 2.50, "read": 0.10}   # Claude Sonnet 5.5, $ per million tokens (read = 0.05x)
MIN_CACHEABLE = 512


def simulate_caching(requests: list[dict]) -> list[tuple[int, int, int]]:
    """Simplified automatic caching: each request reads the longest unchanged prefix of the previous one
    (if >= 512 tokens) and writes the rest. Returns (read, written, total) per request."""
    out, prev = [], None
    for req in requests:
        units = cache_units(req)
        total = sum(est_tokens(u) for u in units)
        read = unchanged_prefix(prev, req) if prev else 0
        read = read if read >= MIN_CACHEABLE else 0
        out.append((read, total - read, total))
        prev = req
    return out


def conversation(build, turns: int = 10) -> list[dict]:
    reqs, hist, f = [], [], CaseFacts(customer_id=Fact(value="CUST-58214", source="get_customer", turn=1))
    for t in range(1, turns + 1):
        ingest_claim(f, 1, f"note_{t}", f"customer detail #{t}", turn=t)  # the facts change every turn
        req = build(hist, f.render(), f"Customer message {t}: " + "details " * 40)
        reqs.append(req)
        hist = req["messages"] + [{"role": "assistant", "content": f"Agent reply {t}: " + "answer " * 30}]
    return reqs


for name, build in [("facts in system", in_system), ("facts in newest turn", in_newest_turn)]:
    stats = simulate_caching(conversation(build))
    read, written, total = (sum(s[i] for s in stats) for i in range(3))
    cost = (read * PRICE["read"] + written * PRICE["write_5m"]) / 1e6
    no_cache = total * PRICE["input"] / 1e6
    print(f"{name:<21} cache reads {read / total:>4.0%} of input  | ~${cost:.4f} vs ~${no_cache:.4f} without caching")

> **Observe:** the two designs send the same information. With the facts in `system`, every turn rewrites the cache from the system prompt onward and caching *costs* more than it saves (writes are 1.25×). With the facts in the newest turn, almost the whole prefix is read at 0.05× (the Sonnet 5.5 rate). (This is a simplified model of automatic caching; the live cell below shows the real `usage` fields.)

**(live)** Three turns per design with automatic caching (`cache_control` at the top level). Watch `cache_read_input_tokens`.

In [ ]:
LONG_POLICY = ("Refund policy section: damaged items qualify for a full refund within 30 days of delivery; "
               "store credit is offered only if the customer prefers it; managers approve refunds over $50. ") * 25

for name, build in [("facts in system", in_system), ("facts in newest turn", in_newest_turn)]:
    hist, f = [], CaseFacts(customer_id=Fact(value="CUST-58214", source="get_customer", turn=1))
    for t in range(1, 4):
        ingest_claim(f, 1, f"note_{t}", f"detail {t}", turn=t)
        req = build(hist, f.render(), f"Short question {t}: what's next?")
        req["system"] = req["system"].replace(POLICY, LONG_POLICY)
        r = client.messages.create(model=MODEL, max_tokens=150, cache_control={"type": "ephemeral"}, **req)
        u = r.usage
        print(f"{name:<21} turn {t}: read={u.cache_read_input_tokens or 0:>5} write={u.cache_creation_input_tokens or 0:>5} uncached={u.input_tokens:>5}")
        hist = req["messages"] + [{"role": "assistant", "content": text_of(r) or "ok"}]

## 11.13 Putting it together: progressive discovery vs monolithic context

**Monolithic context** loads everything up front; **progressive discovery** starts lean and fetches what each request needs (tools, subagents, memory, skills). Most production systems combine them: a cached, monolithic stable prefix for instructions and policy, plus progressive discovery for everything case-specific.

[Theory](README.md#1113-putting-it-together-progressive-discovery-vs-monolithic-context)

In [ ]:
def per_request(prefix_tokens: int, fetched_tokens: int, conversation_tokens: int, cached: bool) -> float:
    """$ per request on Sonnet 5.5 with a warm cache on the prefix (if cached)."""
    prefix_cost = prefix_tokens * (PRICE["read"] if cached else PRICE["input"])
    return (prefix_cost + (fetched_tokens + conversation_tokens) * PRICE["input"]) / 1e6


designs = {
    "monolithic (all manuals + all history)": dict(prefix_tokens=300_000, fetched_tokens=0, conversation_tokens=8_000, cached=True),
    "progressive only (nothing cached)": dict(prefix_tokens=4_000, fetched_tokens=3_000, conversation_tokens=8_000, cached=False),
    "hybrid: cached core + fetched slice": dict(prefix_tokens=4_000, fetched_tokens=3_000, conversation_tokens=8_000, cached=True),
}
for name, d in designs.items():
    window = d["prefix_tokens"] + d["fetched_tokens"] + d["conversation_tokens"]
    print(f"{name:<40} window ~{window:>7,} tokens  ~${per_request(**d):.4f}/request")


def choose_context_strategy(corpus_tokens: int, share_needed_per_request: float, changes_often: bool) -> str:
    if corpus_tokens <= 50_000 and share_needed_per_request > 0.5 and not changes_often:
        return "monolithic (cache it)"
    return "hybrid: cache the stable core, fetch case-specific data just in time"


print()
print("20K-token FAQ, most of it used every time  ->", choose_context_strategy(20_000, 0.8, False))
print("3M-token knowledge base, 1% per question   ->", choose_context_strategy(3_000_000, 0.01, True))

> **Observe:** even with a warm cache, the monolithic design fills a third of the window on every request (context rot and lost-in-the-middle risk) and still costs more. The hybrid keeps the window small *and* cheap. Its risk is a retrieval miss, which you manage with good tools and the facts block.

## Mini-project: a long-running support conversation manager

Combine the chapter into one component and measure it. A scripted **40-turn, two-issue** support case (the running example) is replayed through three designs:

| Variant | History | Tool results | Facts | Compaction |
|---|---|---|---|---|
| `naive` | everything verbatim | raw (43 fields) | none | none |
| `summary_only` | summary + recent turns | raw | none | when over budget |
| `managed` | summary + recent turns | **trimmed at ingest** | **case-facts block in the newest turn** | when over budget, **facts stay outside** |

The `managed` variant also **checkpoints** after every turn (atomic JSON) and is **crashed and restored** halfway. At the end you check (1) tokens per turn, (2) which exact values survive in the final request, and (3) that the restored run sends exactly the same request as an uninterrupted one.

The offline summarizer below is deliberately pessimistic: it keeps topics and drops values, the failure mode from 11.1. In production you would call server-side compaction (11.9) with instructions to keep values, and keep the facts block outside the summary anyway.

In [ ]:
# ---- the scripted case: (customer text, optional tool call, extracted claims) per turn ----
FILLER_TURNS = [
    "Thanks. How long does approval usually take?", "Can I get an email confirmation?", "Is there a way to speed it up?",
    "Okay. The box was also soaked, does that matter?", "Should I keep the packaging?", "I uploaded the photos now.",
    "Did the photos come through?", "Fine, I'll wait.", "Any news from the manager?", "Can you check again?",
    "What does 'in review' mean exactly?", "I'm going on a trip next week, will that be a problem?",
    "Please note I prefer email over phone.", "Is the courier picking up the blender?", "When will the courier come?",
    "Can I drop it at a store instead?", "Which store is closest to Springfield?", "Okay, thanks for checking.",
    "Still nothing from the manager?", "Is there anything else you need from me?", "Got it.", "I'll check my inbox.",
    "One more thing about the kettle delivery.", "Has the kettle been rerouted yet?", "Can I track it?",
    "Thanks for the tracking info.", "The tracking page shows nothing new.", "Okay, it moved to the hub.",
    "Will I get a notification when the refund is paid?", "How many days until the money is back on my card?",
    "Can you add a note that I was patient?", "Alright, thank you.",
]
CASE: list[dict] = [
    {"say": "Hi, I have two problems with recent orders. My customer ID is CUST-58214."},
    {"say": "First: the blender from ORD-30417 arrived damaged.", "tool": ("ORD-30417", 1)},
    {"say": "I want a full refund of $74.95, not store credit.", "claims": [(1, "customer_request", "full refund, NOT store credit")]},
    {"say": "It was ordered on 2025-01-15, so it's within 30 days of delivery."},
    {"say": "Also, your colleague promised me a 25% discount on my next order.", "claims": [(1, "promised", "25% discount on next order")]},
    {"say": "Second issue: order ORD-67911 is going to my old address.", "tool": ("ORD-67911", 2)},
    {"say": "My new address is 42 Elm Street, Springfield.", "claims": [(2, "new_address", "42 Elm Street, Springfield")]},
]
for i, s in enumerate(FILLER_TURNS):
    turn = {"say": s}
    if i % 2 == 0:  # agents re-check orders often: 16 more lookup_order calls
        turn["tool"] = ("ORD-30417", 1) if i % 4 == 0 else ("ORD-67911", 2)
    CASE.append(turn)
CASE.append({"say": "So, to confirm: exactly what are you doing about my refund, my discount and my kettle's address?"})
assert len(CASE) == 40  # the last turn is the question that needs every early value
print(len(CASE), "turns;", sum("tool" in t for t in CASE), "lookup_order calls")

MUST_SURVIVE = ["ORD-30417", "74.95", "2025-01-15", "full refund", "store credit", "25%", "ORD-67911", "42 Elm Street"]

In [ ]:
def lossy_summary(old_turns: list[list[dict]]) -> str:
    """Offline stand-in for a summarizer: keeps topics, drops values (the failure mode from 11.1)."""
    blob = json.dumps(old_turns).lower()
    topics = [t for t, kw in [("a damaged item and compensation", "damaged"), ("a discount", "discount"),
                              ("a delivery address problem", "address"), ("manager approval", "manager")] if kw in blob]
    return f"Earlier in the conversation ({len(old_turns)} turns) the customer discussed " + ", ".join(topics) + "."


class SupportManager:
    """Builds every request from explicit layers: cached policy | summary | recent turns | facts | question."""

    def __init__(self, variant: str, compact_at: int = 6_000, keep_recent: int = 4):
        self.variant, self.compact_at, self.keep_recent = variant, compact_at, keep_recent
        self.turn_no = 0
        self.turns: list[list[dict]] = []      # each turn = the messages it added, as sent (append-only)
        self.summary: str | None = None
        self.facts = CaseFacts()
        self.raw_store: dict[str, dict] = {}   # raw tool payloads live outside the window

    # ---- one customer turn -----------------------------------------------------------
    def handle(self, step: dict) -> dict:
        self.turn_no += 1
        managed = self.variant == "managed"
        if self.turn_no == 1 and managed:
            self.facts.customer_id = Fact(value="CUST-58214", source="get_customer", turn=1)
        for issue_id, key, value in step.get("claims", []):
            if managed:
                ingest_claim(self.facts, issue_id, key, value, self.turn_no)
        user_content = [text(step["say"])]
        if managed:  # the newest copy of the facts goes in the newest turn (append-only, cache-friendly)
            user_content = [text(self.facts.render() + "\n(The latest case_facts block supersedes earlier ones.)")] + user_content
        msgs = [{"role": "user", "content": user_content}]
        if "tool" in step:
            order_id, issue_id = step["tool"]
            raw_order = ORDERS[order_id]
            tid = f"toolu_{self.turn_no:03d}"
            if managed:
                self.raw_store[order_id] = raw_order
                ingest_order(self.facts, issue_id, raw_order, self.turn_no)
                payload = {**{k: raw_order[k] for k in RETURN_FIELDS}, "full_record_ref": f"order_raw/{order_id}"}
            else:
                payload = raw_order
            msgs += [{"role": "assistant", "content": [tool_use("lookup_order", {"order_id": order_id}, id=tid)]},
                     {"role": "user", "content": [{"type": "tool_result", "tool_use_id": tid, "content": json.dumps(payload)}]}]
        request = self.build_request(pending=msgs)
        msgs.append({"role": "assistant", "content": f"(agent reply to turn {self.turn_no})"})  # scripted model reply
        self.turns.append(msgs)
        if self.variant != "naive" and est_tokens(self.build_request()["messages"]) > self.compact_at:
            self.compact()
        return request

    def compact(self) -> None:
        old, self.turns = self.turns[:-self.keep_recent], self.turns[-self.keep_recent:]
        prior = [[{"role": "user", "content": self.summary}]] if self.summary else []
        self.summary = lossy_summary(prior + old)   # facts are NOT in here, and do not need to be

    # ---- the request ---------------------------------------------------------------------
    def build_request(self, pending: list[dict] | None = None) -> dict:
        msgs = [m for t in self.turns for m in t] + (pending or [])
        if self.summary:
            first = msgs[0]
            msgs = [{"role": "user", "content": [text(f"<conversation_summary>\n{self.summary}\n</conversation_summary>")]
                     + list(first["content"])}] + msgs[1:]
        return {"model": MODEL, "max_tokens": 1024, "tools": TOOLS,
                "system": [{"type": "text", "text": SUPPORT_SYSTEM, "cache_control": {"type": "ephemeral"}}],
                "messages": msgs}

    # ---- checkpoint / restore --------------------------------------------------------------
    def checkpoint(self, path: Path) -> None:
        write_json_atomic(path, {"schema_version": 1, "variant": self.variant, "turn_no": self.turn_no,
                                 "turns": self.turns, "summary": self.summary, "facts": self.facts.model_dump(),
                                 "raw_store": self.raw_store, "compact_at": self.compact_at, "keep_recent": self.keep_recent})

    @classmethod
    def restore(cls, path: Path) -> "SupportManager":
        s = json.loads(path.read_text())
        m = cls(s["variant"], s["compact_at"], s["keep_recent"])
        m.turn_no, m.turns, m.summary, m.raw_store = s["turn_no"], s["turns"], s["summary"], s["raw_store"]
        m.facts = CaseFacts.model_validate(s["facts"])
        return m


print("SupportManager ready")

### Token counting: live if possible, labeled estimates otherwise

`count_request` uses the real `count_tokens` endpoint when a working key is available (one probe call decides), and falls back to the offline estimate otherwise. Every number printed says which one it is.

In [ ]:
def _probe_live() -> bool:
    try:
        client.messages.count_tokens(model=MODEL, messages=[{"role": "user", "content": "ping"}])
        return True
    except (anthropic.APIError, TypeError):   # no key, fake key, no network
        return False


LIVE_COUNT = _probe_live()
COUNT_LABEL = "count_tokens" if LIVE_COUNT else "offline estimate"


def count_request(req: dict) -> int:
    if LIVE_COUNT:
        return client.messages.count_tokens(model=req["model"], system=req["system"], tools=req["tools"],
                                            messages=req["messages"]).input_tokens
    return est_tokens([req["system"], req["tools"], req["messages"]])


print("token numbers below come from:", COUNT_LABEL)

In [ ]:
CKPT = WORKDIR / "support-checkpoint.json"
CRASH_AFTER = 25
SAMPLE_TURNS = {1, 5, 10, 15, 20, 25, 30, 35, 40}

results: dict[str, dict] = {}
for variant in ["naive", "summary_only", "managed"]:
    mgr, per_turn = SupportManager(variant), {}
    for n, step in enumerate(CASE, 1):
        req = mgr.handle(step)
        if n in SAMPLE_TURNS:
            per_turn[n] = count_request(req)
        if variant == "managed":
            mgr.checkpoint(CKPT)                      # after EVERY turn
            if n == CRASH_AFTER:
                before_crash = mgr.build_request()
                del mgr                               # the process dies...
                mgr = SupportManager.restore(CKPT)    # ...and a new one restores from disk
                assert mgr.build_request() == before_crash, "restore must resend exactly what was stored"
    results[variant] = {"per_turn": per_turn, "final": req}

print(f"tokens per request ({COUNT_LABEL})")
print("turn  " + "".join(f"{v:>14}" for v in results))
for n in sorted(SAMPLE_TURNS):
    print(f"{n:>4}  " + "".join(f"{results[v]['per_turn'][n]:>14,}" for v in results))
print(" sum  " + "".join(f"{sum(results[v]['per_turn'].values()):>14,}" for v in results), " <- input billed at these turns")

In [ ]:
print("Does the final request (turn 40) still contain the exact values?\n")
print(f"{'value':<16}" + "".join(f"{v:>14}" for v in results))
for value in MUST_SURVIVE:
    row = "".join(f"{'✓' if value in json.dumps(results[v]['final']) else '✗':>14}" for v in results)
    print(f"{value:<16}{row}")

# Crash/restore check: an uninterrupted managed run must produce the same final request
clean = SupportManager("managed")
for step in CASE:
    clean_final = clean.handle(step)
print("\nrestored run sends the same final request as an uninterrupted run:", clean_final == results["managed"]["final"])

> **Observe:** `naive` keeps every value but grows without bound, mostly with raw order data. `summary_only` stays small but loses the customer's own words (the refund type, the 25% promise, the new address); the order ID and amount survive only because a raw `lookup_order` result happens to sit in the last few turns, which is luck, not design. `managed` stays small **and** keeps every value, because the values live in the facts block, tool results entered history already trimmed, and the summary only has to carry the gist. Its curve is not free: each turn carries a fresh facts copy, and older copies stay until compaction removes them (for verified facts, a turn-scoped system message from 11.1 avoids that cost on supported models). The checkpoint restore is exact because the manager stores the turns *as sent* and rebuilds the request from them.

> **Try it:** (1) set `keep_recent=8` and watch the token curve; (2) remove the facts block from `handle` and rerun the survival table; (3) replace `lossy_summary` with a server-side compaction call (11.9) and check which values the real summary keeps.

**(live)** Send the managed final request and read the answer.

In [ ]:
final_req = dict(results["managed"]["final"])
r = client.messages.create(**final_req)
print(r.stop_reason, "| usage:", r.usage.input_tokens, "in,", r.usage.output_tokens, "out")
print(text_of(r)[:800])

## Recap

- The API is stateless; context is something you **design**: a cached static prefix, a summary, recent turns, trimmed tool results, a facts block and the question, each with its own rule.
- Exact values live in a **case-facts block** (per issue, with source, turn and conflict flags), sent in the newest turn (verified facts may use a turn-scoped system message), never in a summary.
- **Trim tool results at ingest** and keep raw payloads outside the window; never edit old turns in place on current models. Fix verbose upstream agents with an output contract.
- Long inputs: **key findings first, headers for details, task last**, and required metadata from subagents.
- Long work: **scratchpads with evidence**, **subagents** for verbose exploration, **atomic state files + a manifest** for crash recovery, and a deliberate **resume / fork / fresh** decision.
- Platform features: count with `count_tokens`, measure with all `usage` fields, **compaction** for long conversations, **context editing** for old tool results, the **memory tool** with path validation, and **caching** that rewards append-only layouts.

Next: [homework](02_homework.ipynb)